In [1]:
"""
@author: Zilan Cheng
@note: This code is modified based on Zongyi Li's original implementation of Fourier Neural Operators.
"""

import torch.nn.functional as F
from timeit import default_timer
from utilities3 import *
import numpy as np
import matplotlib.pyplot as plt
torch.cuda.set_device(1)
torch.manual_seed(0)
np.random.seed(0)

In [2]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(SpectralConv2d, self).__init__()

        """
        2D Fourier layer. It does FFT, linear transform, and Inverse FFT.    
        """

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2

        self.scale = (1 / (in_channels * out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft2(x)

        out_ft = torch.zeros(batchsize, self.out_channels,  x.size(-2), x.size(-1)//2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)

        x = torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))
        return x

In [3]:
class MLP(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels):
        super(MLP, self).__init__()
        self.mlp1 = nn.Conv2d(in_channels, mid_channels, 1)
        self.mlp2 = nn.Conv2d(mid_channels, out_channels, 1)

    def forward(self, x):
        x = self.mlp1(x)
        x = F.gelu(x)
        x = self.mlp2(x)
        return x

In [4]:
def get_grid(shape, device):
    batchsize, size_x, size_y = shape[0], shape[1], shape[2]
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1).repeat([batchsize, 1, size_y, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1).repeat([batchsize, size_x, 1, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)

In [5]:
class FNO2d(nn.Module):
    def __init__(self, modes1, modes2,  width):
        super(FNO2d, self).__init__()

        self.modes1 = modes1
        self.modes2 = modes2
        self.width = width
        self.padding = 9 # pad the domain if input is non-periodic

        self.p = nn.Linear(3, self.width) # input channel is 3: (a(x, y), x, y)
        self.conv0 = SpectralConv2d(self.width, self.width, self.modes1, self.modes2)
        self.conv1 = SpectralConv2d(self.width, self.width, self.modes1, self.modes2)
        self.conv2 = SpectralConv2d(self.width, self.width, self.modes1, self.modes2)
        self.conv3 = SpectralConv2d(self.width, self.width, self.modes1, self.modes2)
        self.mlp0 = MLP(self.width, self.width, self.width)
        self.mlp1 = MLP(self.width, self.width, self.width)
        self.mlp2 = MLP(self.width, self.width, self.width)
        self.mlp3 = MLP(self.width, self.width, self.width)
        self.w0 = nn.Conv2d(self.width, self.width, 1)
        self.w1 = nn.Conv2d(self.width, self.width, 1)
        self.w2 = nn.Conv2d(self.width, self.width, 1)
        self.w3 = nn.Conv2d(self.width, self.width, 1)
        self.q = MLP(self.width, 1, self.width * 4) # output channel is 1: u(x, y)

    def forward(self, x):
        grid = get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x=x.to(torch.float32)
        x = self.p(x)
        x = x.permute(0, 3, 1, 2)
        x = F.pad(x, [0,self.padding, 0,self.padding])

        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x1 = self.mlp3(x1)
        x2 = self.w3(x)
        x = x1 + x2

        x = x[..., :-self.padding, :-self.padding]
        x = self.q(x)
        x = x.permute(0, 2, 3, 1)
        return x

In [6]:
################################################################
# configs
################################################################
ntrain = 900
ntest = 100

width = 32
modes=3

s=256
r=2

batch_size = 20
learning_rate = 0.001
epochs = 2000
iterations = epochs*(ntrain//batch_size)

In [7]:
################################################################
# dataloader
################################################################
u_end=np.load("../data/kp/u_end_512.npy")
u_end=torch.tensor(u_end)
u_end=u_end.permute(2,0,1)
u1=np.load("../data/kp/u1_512.npy")
u1=torch.tensor(u1)
u1=u1.permute(2,0,1)

x_train=u1[:ntrain,:,:][:,::r,::r]
x_test=u1[-ntest:,:,:][:,::r,::r]
y_train=u_end[:ntrain,:,:][:,::r,::r]
y_test=u_end[-ntest:,:,:][:,::r,::r]

x_train = x_train.reshape(ntrain,s,s,1)
x_test = x_test.reshape(ntest,s,s,1)
y_train = y_train.reshape(ntrain,s,s,1)
y_test = y_test.reshape(ntest,s,s,1)

x_normalizer = UnitGaussianNormalizer(x_train)
x_train = x_normalizer.encode(x_train)
x_test = x_normalizer.encode(x_test)

y_normalizer = UnitGaussianNormalizer(y_train)
y_train = y_normalizer.encode(y_train)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)

In [8]:
solution_real=torch.zeros(ntest,s,s,1)
solution_learnt=torch.zeros(ntest,s,s,1)

In [9]:
################################################################
# training and evaluation
################################################################
model = FNO2d(modes, modes, width).to(device) 
print(count_params(model))

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)

myloss = LpLoss(size_average=True)
y_normalizer.to(device)
for ep in range(epochs):
    i=0
    model.train()
    train_l2 = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x).reshape(batch_size, s, s,1)
        out = y_normalizer.decode(out)
        y = y_normalizer.decode(y)

        loss = myloss(out.view(batch_size,-1), y.view(batch_size,-1))
        loss.backward()

        optimizer.step()
        scheduler.step()
        train_l2 += loss.item()
    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x).reshape(batch_size, s, s,1)
            out = y_normalizer.decode(out)
            solution_real[i:i+batch_size,:,:,:]=y
            solution_learnt[i:i+batch_size,:,:,:]=out
            i=i+batch_size
            test_l2 += myloss(out.view(batch_size,-1), y.view(batch_size,-1)).item()
    train_l2/= ntrain/batch_size
    test_l2 /= ntest/batch_size

    if ep % 50 == 0 or ep == epochs - 1:
        print(f"Epoch {ep:4d} | Train L2: {train_l2:.6f} | Test L2: {test_l2:.6f}")

164609
Epoch    0 | Train L2: 0.371907 | Test L2: 0.309925
Epoch   50 | Train L2: 0.136413 | Test L2: 0.133312
Epoch  100 | Train L2: 0.124579 | Test L2: 0.106572
Epoch  150 | Train L2: 0.100004 | Test L2: 0.102876
Epoch  200 | Train L2: 0.083995 | Test L2: 0.088288
Epoch  250 | Train L2: 0.076587 | Test L2: 0.072509
Epoch  300 | Train L2: 0.091863 | Test L2: 0.089566
Epoch  350 | Train L2: 0.071575 | Test L2: 0.067120
Epoch  400 | Train L2: 0.060841 | Test L2: 0.071117
Epoch  450 | Train L2: 0.057991 | Test L2: 0.061404
Epoch  500 | Train L2: 0.074367 | Test L2: 0.059380
Epoch  550 | Train L2: 0.062733 | Test L2: 0.060865
Epoch  600 | Train L2: 0.070040 | Test L2: 0.061828
Epoch  650 | Train L2: 0.057306 | Test L2: 0.061272
Epoch  700 | Train L2: 0.056649 | Test L2: 0.052621
Epoch  750 | Train L2: 0.048540 | Test L2: 0.055385
Epoch  800 | Train L2: 0.049353 | Test L2: 0.053598
Epoch  850 | Train L2: 0.057579 | Test L2: 0.055634
Epoch  900 | Train L2: 0.049117 | Test L2: 0.046662
Epoch